# 02 — Fetch The Odds API historical odds

This notebook uses The Odds API credits.

Default setup:
- World Cup 2022;
- `h2h` market;
- `eu` region;
- snapshots: T-24h, T-3h, T-1h;
- maximum 64 events;
- expected cost is about 1,950 credits.

Output:
- `data/processed/historical_odds_aggregated.csv`
- `data/processed/02_historical_odds_report_bundle.zip`


In [ ]:
# Cell 1. Paste The Odds API key here.

ODDS_API_KEY = "PASTE_THE_ODDS_API_KEY_HERE"


In [ ]:
# Cell 2. Imports, helpers, API helpers.


from pathlib import Path
import json
import re
import zipfile
import time

import numpy as np
import pandas as pd

DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP_UTC = pd.Timestamp.utcnow().isoformat()
SEED = 42

TEAM_REPLACEMENTS = {
    "USA": "United States",
    "US": "United States",
    "Korea Republic": "South Korea",
    "Türkiye": "Turkey",
    "Czechia": "Czech Republic",
    "Bosnia and Herzegovina": "Bosnia & Herzegovina",
    "Ivory Coast": "Côte d'Ivoire",
    "Cote d Ivoire": "Côte d'Ivoire",
}

def norm_team(name):
    if pd.isna(name):
        return ""
    name = str(name).strip()
    name = re.sub(r"\s+", " ", name)
    return TEAM_REPLACEMENTS.get(name, name)

def save_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(
            payload,
            f,
            indent=2,
            ensure_ascii=False,
            default=str,
        )
    return path

def infer_outcome(home_goals, away_goals):
    if pd.isna(home_goals) or pd.isna(away_goals):
        return np.nan
    if home_goals > away_goals:
        return 2
    if home_goals < away_goals:
        return 0
    return 1

def expected_score(rating_a, rating_b):
    return 1.0 / (1.0 + 10 ** (-(rating_a - rating_b) / 400.0))



import requests

ODDS_BASE = "https://api.the-odds-api.com/v4"

def has_key(value):
    if value is None:
        return False
    value = str(value).strip()
    if value == "":
        return False
    if value.startswith("PASTE_"):
        return False
    return True

def mask_key(value):
    if not has_key(value):
        return "missing"
    value = str(value)
    return f"present_len_{len(value)}_last4_{value[-4:]}"

def safe_get_json(url, params=None, timeout=60):
    try:
        response = requests.get(
            url,
            params=params or {},
            timeout=timeout,
        )
        try:
            data = response.json()
        except Exception:
            data = {"raw_text": response.text[:3000]}
        return response.status_code, data, dict(response.headers)
    except Exception as exc:
        return -1, {"error": str(exc)}, {}

def sanitize_params(params):
    out = dict(params or {})
    for key in list(out.keys()):
        if str(key).lower() in {"apikey", "api_key", "key", "token"}:
            out[key] = mask_key(out[key])
    return out

def usage_headers(headers):
    return {
        "x_requests_remaining": headers.get("x-requests-remaining"),
        "x_requests_used": headers.get("x-requests-used"),
        "x_requests_last": headers.get("x-requests-last"),
    }

def add_report_row(
    rows,
    check,
    endpoint,
    params,
    status,
    data,
    headers,
    path,
):
    row_count = 0
    if isinstance(data, list):
        row_count = len(data)
    elif isinstance(data, dict):
        value = data.get("data")
        if isinstance(value, list):
            row_count = len(value)
    error = ""
    if isinstance(data, dict):
        error = str(data.get("message", data.get("error", "")))[:500]
    rows.append({
        "run_timestamp_utc": RUN_TIMESTAMP_UTC,
        "check": check,
        "endpoint": endpoint,
        "params_masked": json.dumps(sanitize_params(params)),
        "status_code": status,
        "rows": row_count,
        "error": error,
        "cache_path": str(path),
        **usage_headers(headers),
    })

def odds_get(
    path,
    params=None,
    name="request",
    out_dir=None,
    report_rows=None,
):
    params = dict(params or {})
    params["apiKey"] = ODDS_API_KEY
    url = f"{ODDS_BASE}/{path.lstrip('/')}"
    status, data, headers = safe_get_json(url, params=params)
    safe_name = re.sub(r"[^a-zA-Z0-9_]+", "_", name)
    if out_dir is not None:
        out_path = Path(out_dir) / "raw" / f"{safe_name}.json"
        save_json(out_path, {
            "url_path": path,
            "params_masked": sanitize_params(params),
            "status_code": status,
            "headers": headers,
            "data": data,
        })
    else:
        out_path = Path(f"{safe_name}.json")
    if report_rows is not None:
        add_report_row(
            report_rows,
            check=name,
            endpoint=path,
            params=params,
            status=status,
            data=data,
            headers=headers,
            path=out_path,
        )
    return status, data, headers


HIST_DIR = PROCESSED_DIR / "historical_odds"
HIST_DIR.mkdir(parents=True, exist_ok=True)

if not has_key(ODDS_API_KEY):
    raise ValueError("Paste ODDS_API_KEY in Cell 1 first.")

print("Setup OK:", ODDS_BASE)


In [ ]:
# Cell 3. Config and credit guard.

SPORT_KEY = "soccer_fifa_world_cup"
REGION = "eu"
MARKETS = "h2h"

DISCOVERY_DATES = pd.date_range(
    "2022-11-20 12:00:00",
    "2022-12-18 12:00:00",
    freq="2D",
    tz="UTC",
)

SNAPSHOT_OFFSETS_HOURS = [
    24,
    3,
    1,
]

MAX_EVENTS = 64
MAX_CREDITS_THIS_RUN = 2500

estimated_credits = (
    len(DISCOVERY_DATES) * 1
    + MAX_EVENTS * len(SNAPSHOT_OFFSETS_HOURS) * 10
)

budget = pd.DataFrame([{
    "max_credits_this_run": MAX_CREDITS_THIS_RUN,
    "estimated_credits": estimated_credits,
    "discovery_calls": len(DISCOVERY_DATES),
    "max_events": MAX_EVENTS,
    "snapshots": str(SNAPSHOT_OFFSETS_HOURS),
    "region": REGION,
    "markets": MARKETS,
}])

budget.to_csv(HIST_DIR / "historical_odds_budget.csv", index=False)

display(budget)

if estimated_credits > MAX_CREDITS_THIS_RUN:
    raise ValueError("Estimated credits exceed MAX_CREDITS_THIS_RUN.")


In [ ]:
# Cell 4. Discover World Cup 2022 historical events.

report_rows = []
event_rows = []

for ts in DISCOVERY_DATES:
    ts_iso = ts.isoformat().replace("+00:00", "Z")

    status, data, headers = odds_get(
        f"historical/sports/{SPORT_KEY}/events",
        params={
            "dateFormat": "iso",
            "date": ts_iso,
        },
        name=f"events_{ts_iso}",
        out_dir=HIST_DIR,
        report_rows=report_rows,
    )

    events = []
    if isinstance(data, dict):
        events = data.get("data", []) or []

    for event in events:
        event_rows.append({
            "sport_key": SPORT_KEY,
            "event_id": event.get("id"),
            "commence_time": event.get("commence_time"),
            "home_team": norm_team(event.get("home_team")),
            "away_team": norm_team(event.get("away_team")),
            "discovery_timestamp": ts_iso,
        })

    time.sleep(0.15)

events_df = pd.DataFrame(event_rows)

if len(events_df) > 0:
    events_df = events_df.drop_duplicates(["sport_key", "event_id"])
    events_df["commence_time"] = pd.to_datetime(
        events_df["commence_time"],
        utc=True,
        errors="coerce",
    )
    events_df = events_df.sort_values("commence_time").head(MAX_EVENTS)

events_df.to_csv(
    HIST_DIR / "historical_events_discovered.csv",
    index=False,
)

display(events_df)
print("Events:", len(events_df))


In [ ]:
# Cell 5. Fetch event-level historical odds.

def parse_event_odds(
    event_id,
    snapshot_label,
    snapshot_timestamp,
    data,
):
    rows = []

    if not isinstance(data, dict):
        return rows

    event_obj = data.get("data", data)
    events = event_obj if isinstance(event_obj, list) else [event_obj]

    for event in events:
        home_team = norm_team(event.get("home_team", ""))
        away_team = norm_team(event.get("away_team", ""))
        commence_time = event.get("commence_time")

        for bookmaker in event.get("bookmakers", []) or []:
            bookmaker_key = bookmaker.get("key")
            bookmaker_title = bookmaker.get("title")

            for market in bookmaker.get("markets", []) or []:
                if market.get("key") != "h2h":
                    continue

                prices = {}

                for outcome in market.get("outcomes", []) or []:
                    name = norm_team(outcome.get("name", ""))
                    price = pd.to_numeric(
                        outcome.get("price"),
                        errors="coerce",
                    )

                    if pd.isna(price):
                        continue

                    if name == home_team:
                        prices["home_odds"] = float(price)
                    elif name == away_team:
                        prices["away_odds"] = float(price)
                    elif str(name).lower() in {"draw", "tie"}:
                        prices["draw_odds"] = float(price)

                if {"home_odds", "draw_odds", "away_odds"} <= set(prices):
                    rows.append({
                        "event_id": event_id,
                        "snapshot_label": snapshot_label,
                        "snapshot_timestamp": snapshot_timestamp,
                        "commence_time": commence_time,
                        "home_team": home_team,
                        "away_team": away_team,
                        "bookmaker_key": bookmaker_key,
                        "bookmaker_title": bookmaker_title,
                        **prices,
                    })

    return rows

odds_rows = []

for _, event in events_df.iterrows():
    event_id = event["event_id"]
    kickoff = event["commence_time"]

    if pd.isna(kickoff):
        continue

    for offset in SNAPSHOT_OFFSETS_HOURS:
        snap_ts = kickoff - pd.Timedelta(hours=offset)
        snap_iso = snap_ts.isoformat().replace("+00:00", "Z")
        snap_label = f"T_minus_{offset}h"

        status, data, headers = odds_get(
            f"historical/sports/{SPORT_KEY}/events/{event_id}/odds",
            params={
                "regions": REGION,
                "markets": MARKETS,
                "oddsFormat": "decimal",
                "dateFormat": "iso",
                "date": snap_iso,
            },
            name=f"event_{event_id}_{snap_label}",
            out_dir=HIST_DIR,
            report_rows=report_rows,
        )

        odds_rows.extend(
            parse_event_odds(
                event_id=event_id,
                snapshot_label=snap_label,
                snapshot_timestamp=snap_iso,
                data=data,
            )
        )

        time.sleep(0.12)

historical_odds_raw = pd.DataFrame(odds_rows)

historical_odds_raw.to_csv(
    HIST_DIR / "historical_odds_raw_bookmaker_rows.csv",
    index=False,
)

display(historical_odds_raw.head(30))
print("Raw odds rows:", len(historical_odds_raw))


In [ ]:
# Cell 6. Aggregate historical odds.

if len(historical_odds_raw) == 0:
    historical_odds_agg = pd.DataFrame()
else:
    rows = []

    group_cols = [
        "event_id",
        "snapshot_label",
        "snapshot_timestamp",
        "commence_time",
        "home_team",
        "away_team",
    ]

    for keys, group in historical_odds_raw.groupby(group_cols):
        row = dict(zip(group_cols, keys))

        avg_home = group["home_odds"].mean()
        avg_draw = group["draw_odds"].mean()
        avg_away = group["away_odds"].mean()

        raw = np.array([
            1.0 / avg_away,
            1.0 / avg_draw,
            1.0 / avg_home,
        ])

        probs = raw / raw.sum()

        row.update({
            "n_bookmakers": group["bookmaker_key"].nunique(),
            "avg_home_odds": float(avg_home),
            "avg_draw_odds": float(avg_draw),
            "avg_away_odds": float(avg_away),
            "best_home_odds": float(group["home_odds"].max()),
            "best_draw_odds": float(group["draw_odds"].max()),
            "best_away_odds": float(group["away_odds"].max()),
            "market_margin_avg": float(raw.sum() - 1.0),
            "market_p_away_devig": float(probs[0]),
            "market_p_draw_devig": float(probs[1]),
            "market_p_home_devig": float(probs[2]),
        })

        rows.append(row)

    historical_odds_agg = pd.DataFrame(rows)

historical_odds_agg.to_csv(
    HIST_DIR / "historical_odds_aggregated.csv",
    index=False,
)

# Also copy to processed root for next notebooks.
historical_odds_agg.to_csv(
    PROCESSED_DIR / "historical_odds_aggregated.csv",
    index=False,
)

display(historical_odds_agg.head(30))
print("Aggregated snapshots:", len(historical_odds_agg))


In [ ]:
# Cell 7. Save historical odds report.

status_df = pd.DataFrame(report_rows)

status_df.to_csv(
    HIST_DIR / "historical_odds_status.csv",
    index=False,
)

actual_credit_sum = (
    pd.to_numeric(
        status_df["x_requests_last"],
        errors="coerce",
    ).fillna(0).sum()
    if "x_requests_last" in status_df.columns
    else None
)

summary = {
    "run_timestamp_utc": RUN_TIMESTAMP_UTC,
    "events": int(len(events_df)),
    "raw_odds_rows": int(len(historical_odds_raw)),
    "aggregated_snapshots": int(len(historical_odds_agg)),
    "estimated_credits": int(estimated_credits),
    "actual_credit_sum_from_headers": actual_credit_sum,
    "region": REGION,
    "markets": MARKETS,
    "snapshot_offsets_hours": SNAPSHOT_OFFSETS_HOURS,
}

save_json(HIST_DIR / "historical_odds_summary.json", summary)

zip_path = PROCESSED_DIR / "02_historical_odds_report_bundle.zip"

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for file in HIST_DIR.rglob("*"):
        if file.is_file():
            zf.write(file, arcname=file.relative_to(HIST_DIR))
    zf.write(
        PROCESSED_DIR / "historical_odds_aggregated.csv",
        arcname="historical_odds_aggregated.csv",
    )

display(status_df.tail(20))
print("Created:", zip_path.resolve())
